# Predictive maintenance — baseline RCF

**Scop.** Un baseline local pentru detectorul de anomalii, înainte de RCF din SageMaker.
`IsolationForest` are același contract de intrare și ieșire ca RCF-ul și rulează pe laptop.

**Intrare.** `ml/artifacts/pm/features.parquet`, produs de `pm-01-dataset.ipynb`.
**Ieșiri.** `rcf-baseline.joblib`, `rcf-scores.csv`, `rcf-baseline.json`.

**Atenție la scurgere.** Intrarea modelului este `MODEL_FEATURE_ORDER` din Lambda: doar
măsurători. `tool_wear_index`, `joint_friction_index` și etichetele rămân în afara vectorului.


## 1. Mediu și dependențe

In [ ]:
from pathlib import Path
import json
import sys

ROOT = Path.cwd().resolve()
while not (ROOT / "ml" / "src").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "ml" / "src").is_dir():
    raise RuntimeError("rulează notebook-ul din repository")
sys.path.insert(0, str(ROOT / "ml" / "src"))
print("repository:", ROOT)

import importlib.util

missing = [name for name in ("pandas", "pyarrow", "sklearn") if importlib.util.find_spec(name) is None]
print("pachete lipsă:", missing)
if missing:
    raise SystemExit('rulează: pip install -e "ml[data,train]"')

import joblib
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.metrics import classification_report, confusion_matrix


## 2. Încarcă tabelul de caracteristici

In [ ]:
ARTIFACTS = ROOT / "ml" / "artifacts" / "pm"
FEATURES_PATH = ARTIFACTS / "features.parquet"
if not FEATURES_PATH.is_file():
    raise SystemExit("rulează mai întâi pm-01-dataset.ipynb")
features = pd.read_parquet(FEATURES_PATH)
print(features.shape)


## 3. Coloanele de intrare

Lista vine din Lambda, ca antrenarea și hot path-ul să vadă aceleași măsurători. Orice coloană
de etichetă ori indice intern al simulării este refuzată aici.

Dacă o coloană din listă lipsește din tabel, notebook-ul o raportează și continuă cu
intersecția. `axis_feed_mean` este astăzi în `MODEL_FEATURE_ORDER`, dar nu în
`ENRICHED_COLUMNS`; până la un bump de contract, baseline-ul folosește doar coloanele produse
efectiv. Nu promova modelul într-un endpoint până când contractul și lista nu coincid.


In [ ]:
sys.path.insert(0, str(ROOT / "lambdas" / "feature_engineering"))
from core import MODEL_FEATURE_ORDER

FEATURES = [name for name in MODEL_FEATURE_ORDER if name in features.columns]
missing_columns = [name for name in MODEL_FEATURE_ORDER if name not in features.columns]
print("coloane lipsă:", missing_columns)
if missing_columns:
    print("ATENȚIE: contractul nu produce toate intrările modelului; vezi nota din markdown")
assert FEATURES
assert not set(FEATURES) & {"tool_wear_index", "joint_friction_index", "health_label", "family", "session"}
print(FEATURES)


## 4. Antrenare pe ferestre sănătoase

`healthy` vine din `pm-01-dataset.ipynb` și folosește eticheta calibrată pe MuJoCo. Când
rularea nu are destule ferestre normale, notebook-ul poate folosi provizoriu cele mai puțin
uzate 20%, ca să valideze pipeline-ul. Cifrele acelea nu susțin un model real; baza normală
trebuie colectată separat.


In [ ]:
healthy = features["healthy"] if "healthy" in features.columns else features["health_label"] == "healthy"
print("ferestre sănătoase:", int(healthy.sum()), "din", len(features))
if int(healthy.sum()) < 10:
    print("prea puține ferestre sănătoase; folosesc provizoriu cele mai puțin uzate 20%")
    wear_column = "tool_wear_index_max" if "tool_wear_index_max" in features.columns else "tool_wear_index"
    cutoff = features[wear_column].quantile(0.2)
    healthy = features[wear_column] <= cutoff
    print("prag provizoriu de uzură:", round(float(cutoff), 6))

X = features[FEATURES].fillna(0.0)
model = IsolationForest(n_estimators=200, contamination="auto", random_state=42)
model.fit(X[healthy])

scores = -model.score_samples(X)
threshold = float(pd.Series(scores[healthy]).quantile(0.99))
features["anomaly_score"] = scores
features["anomaly_flag"] = scores > threshold
print("prag (percentila 99 pe sănătos):", round(threshold, 6))


## 5. Evaluare pe ferestre

Matricea de confuzie tratează orice etichetă diferită de `healthy` ca degradare. Evaluarea pe
episoade, cu preaviz în secunde, este pasul următor; aici verificăm doar că semnalul separat
există.


In [ ]:
truth = ~features["healthy"] if "healthy" in features.columns else features["health_label"] != "healthy"
print(confusion_matrix(truth, features["anomaly_flag"]))
print(classification_report(truth, features["anomaly_flag"], target_names=["sănătos", "degradat"], zero_division=0))


## 6. Scrie scorurile și modelul

In [ ]:
scored = features[["session", "window_start_ms", "health_label", "anomaly_score", "anomaly_flag"]]
scored.to_csv(ARTIFACTS / "rcf-scores.csv", index=False)

joblib.dump({"model": model, "features": FEATURES, "threshold": threshold}, ARTIFACTS / "rcf-baseline.joblib")
metrics = {
    "rows": int(len(features)),
    "healthy_rows": int(healthy.sum()),
    "threshold_p99": threshold,
    "flagged_rows": int(features["anomaly_flag"].sum()),
    "features": FEATURES,
}
(ARTIFACTS / "rcf-baseline.json").write_text(
    json.dumps(metrics, indent=2, ensure_ascii=False) + "\n", encoding="utf-8"
)
print("scris", ARTIFACTS / "rcf-baseline.json")


## 7. Pasul următor

1. Evaluare pe episoade: grupează ferestrele pe `cycle_id` și măsoară preavizul față de
   `events.jsonl`.
2. RCF din SageMaker, cu aceleași coloane și același prag, când baseline-ul este stabil.
3. Mută antrenarea în `ml/training/predictive_maintenance/train.py` și scrie cardul în
   `ml/models/predictive_maintenance.md`.
